### Structured outputs 
- gettong the correct data format for downstream functions etc.
- how we can optimise the rag pipeline to ensure recall and precision scores are increasing as we improve the sytem 
- looking into managing prompts more efficiently - so its not just a hard coded string


what we need running 
- docker compose running 
- http://localhost:8501 (our front end)
- http://localhost:8000/docs (our fastAPI endpoints)
- http://localhost:6333/dashboard#/collections (qdrant db ui)
- https://smith.langchain.com/ (for observability and metrics)


strucured outputs -> many modern llm app are not just one llm call
so the output of one llm is the input to another 
so these have a 'contract schema'
so the outputs need to be correctly formatted 
different llm outputs used in differnet ways downstream 
multiple types of info output by the llm 
i.e lists of data + r easoning 

### Deps

In [19]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

from langsmith import get_current_run_tree

instructor is a library for getting structured outputs from an LLM 
uses things like retries i.e if the format fails initially this is checked and retried with any error traces - i.e where and why it failed 
this is also a context enginerring technique
- retry + provide more context
https://python.useinstructor.com

if we are building agents (not just a single llm call) - other frameworks and wrappers are needed - instructor has some shorcomings

In [20]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Mock Example

In [21]:
prompt = """You are a helpful assistant.
Retutn and answer to the question 
Question: What is your name?"""

In [22]:

response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": prompt}
    ], 
    reasoning_effort="none"
)

print(response.choices[0].message.content)

My name is **ChatGPT**.


In [23]:
response

ChatCompletion(id='chatcmpl-E03uCD63hh9XFyE6eq2LUIksSFipJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='My name is **ChatGPT**.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1783683720, model='gpt-5.4-nano-2026-03-17', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=11, prompt_tokens=28, total_tokens=39, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

the response its self is an object of type 'ChatCompletion'

ChatCompletion(id='chatcmpl-Dxc00yp8cQc2xTuppZHJ3T7hEBiMJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='My name is **ChatGPT**.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1783099792, model='gpt-5.4-nano-2026-03-17', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=11, prompt_tokens=28, total_tokens=39, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

we are extracting the infor we eant from the 'choices' in the above

### Add instructor (structured Outputs)

we previously showed the 'old' method of creating structured outputs using llm calls last week 
lets look at the new way

NOTE: ` mode=instructor.Mode.RESPONSES_TOOLS` newer models can only run toolcall based structured outputs with a pre defined reasoning effort. 

when using instructor lib - a pydantic data object will be returned 
we can define the data classes 


In [24]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [25]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question")

we can structure the context without excessive additions to the input prompt
see below to see the structured output
(this will use more tokens)

In [26]:
response = client.create(
    messages=[
        {"role": "system", "content": prompt}
    ], 
    reasoning={"effort": "none"},
    response_model=Answer
)

In [27]:
response

Answer(answer='I’m ChatGPT, an AI assistant.')

In [28]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ], 
    reasoning={"effort": "none"},
    response_model=Answer
)

this will return two objects  
- the regular response
- the raw response

In [29]:
response

Answer(answer='I’m ChatGPT.')

In [30]:
raw_response

Response(id='resp_0cd565ba6c7de441006a50da8a5b708198b124461f317705c7', created_at=1783683722.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT."}', call_id='call_pqkjAg3rKrAekR7ePKmayVzQ', name='Answer', type='function_call', id='fc_0cd565ba6c7de441006a50da8acd448198a8bf041873ac6cab', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', description='Correctly extracted `Answer` with all the required parameters with correct types')], top_p=0.98, background=False, completed_at=1783683722.0, conversation=

this raw response will give us the parameters used, token usage, etc
(this approach does use more tokens but is worth every penny)

In [31]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="Reasoning for the answer")
    answer: str = Field(description="Answer to the question")

if youre trying to increase the accuracy of the model by using reasoning - ensure the reasoning is added prior to the answer 

In [32]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ], 
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
)

In [33]:
response

AnswerWithReasoning(reasoning='The user asks for my name. As an AI assistant, I don’t have a personal name, but I can provide an assistant name/label to identify myself.', answer='I’m ChatGPT.')

this will show the llms reasoning as to why it came to the answer
you can add any additional data you want from the llm 
i.e check that no unwanted values are being returned
can also add docstringa

### Rag Pipeline

In [34]:
class RAG_GenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [35]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "total_tokens": response.usage.total_tokens
        }

    return response.data[0].embedding

def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-01",
        query=query_embedding, 
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scored = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocess_description"])
        similarity_scored.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])


    return {
        "retrieved_context_ids":  retrieved_context_ids, 
        "retrieved_context": retrieved_context,
        "similarity_scored": similarity_scored,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
    """

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ], 
        reasoning={"effort": "none"},
        response_model=RAG_GenerationResponse
    )

    return response

def rag_pipeline(question, topk_k=5):

    retrieved_context = retrieve_data(question, k=topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer, 
        "question": question, 
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [36]:
output = rag_pipeline("Any usb chargeable devices?")

In [38]:
output

{'data_object': RAG_GenerationResponse(answer='Yes. The available products that are USB-chargeable/USB-powered include:\n\n- HZD Desk Fan Rechargeable (mini portable USB fan) – uses USB power (note: it says it does not come with a battery).\n- Marame 120mm 5V USB Powered Fan – powered by a 3.3ft USB cable.\n- INIU USB-C to USB-C Cable (100W PD) – not a device, but a USB-C charging cable for USB-C devices.\n\nAlso available is ZARIMI Compressed Air Duster – cordless electric with USB charging design.'),
 'answer': 'Yes. The available products that are USB-chargeable/USB-powered include:\n\n- HZD Desk Fan Rechargeable (mini portable USB fan) – uses USB power (note: it says it does not come with a battery).\n- Marame 120mm 5V USB Powered Fan – powered by a 3.3ft USB cable.\n- INIU USB-C to USB-C Cable (100W PD) – not a device, but a USB-C charging cable for USB-C devices.\n\nAlso available is ZARIMI Compressed Air Duster – cordless electric with USB charging design.',
 'question': 'Any us